In [29]:
from typesafe_sdk import AsyncTypeSafeClient, Choice, Noul, Score

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

import pandas as pd

from pathlib import Path

from PIL import Image, ImageDraw
from typesafe_sdk import Box, Image as TSImage, TypeSafeClient

In [30]:
SOURCE = Path("cars.jpg")          # your input image
OUTPUT = Path("cars-boxed.jpg")    # where to save the result
TARGET = "Red car"  # describe the object to box
PAD = 0.0                          # optional padding, as a fraction of the image size

with TypeSafeClient() as client:
    result = client.system_one(
        model="jev-preview",       # Box questions need Jev 1.14
        state={"photo": TSImage(SOURCE)},
        questions={
            "car": Box(instructions=TARGET),
        },
        extra_headers={"TypeSafe-Danger-Zone-Images": "true"},
    )

result


SystemOneResponse(model='jev-1.14.0', usage=Usage(input_tokens=3145, output_tokens=20), answers={'car': BoxAnswer(type='box', box=BoxCoordinates(top=0.67, bottom=0.93, left=0.12, right=0.37), confidence=0.93)})

In [31]:
answer = result.answers["car"]
answer


BoxAnswer(type='box', box=BoxCoordinates(top=0.67, bottom=0.93, left=0.12, right=0.37), confidence=0.93)

In [32]:
edges = answer.box
edges

BoxCoordinates(top=0.67, bottom=0.93, left=0.12, right=0.37)

In [33]:
print(f"Confidence: {answer.confidence:.2f}")

Confidence: 0.93


In [34]:

# Convert fractional edges (0..1) to pixel coordinates
image = Image.open(SOURCE).convert("RGB")
width, height = image.size

left = max(0.0, edges.left - PAD)
top = max(0.0, edges.top - PAD)
right = min(1.0, edges.right + PAD)
bottom = min(1.0, edges.bottom + PAD)

box_px = (
    round(left * width),
    round(top * height),
    round(right * width),
    round(bottom * height),
)

# Draw the bounding box
draw = ImageDraw.Draw(image)
line_width = max(2, round(min(width, height) / 200))
draw.rectangle(box_px, outline=(255, 0, 0), width=line_width)

image.save(OUTPUT, "JPEG", quality=92)
print(f"Saved {OUTPUT} with box at {box_px}")

Saved cars-boxed.jpg with box at (144, 603, 444, 837)
